# 7.1 Performance

FrequenSolve performance is driven by a few practical choices: solve only the frequencies you need, sanity-check in the frequency domain before launching a time-domain sweep, let the solver batch compatible sources internally, and keep receivers inside the same simulation when they sample the same wavefield.

By the end of this tutorial, you should be able to read the run timing metadata, plot assembly and solve phases across a frequency sweep, reason about source batching, and recognize when a modeling change affects physics versus when it only affects sampling.


## The Performance Mental Model

FrequenSolve is primarily a frequency-domain simulator. A frequency-domain job assembles and solves one or more requested frequencies directly. A time-domain job expands a time window and bandwidth into a list of frequency-domain tasks, solves those tasks, and reconstructs time-domain traces from the frequency responses.

That design has two immediate consequences.

| Choice | Performance implication | Practical habit |
| --- | --- | --- |
| Frequency-domain QC first | One or a few frequencies are usually much cheaper than a full time-domain sweep. | Check geometry, boundary conditions, receiver fields, and ParaView output before a long run. |
| Time-domain job | Runtime is roughly the cost of many frequency simulations plus trace reconstruction. | Keep the requested bandwidth and time window scientifically justified. |
| Batched sources | Compatible sources can share setup and solve work. In 2D, internal source batches larger than 64 are split into sub-batches. | Put compatible sources in one acquisition; the solver chooses the execution batches. |
| Multiple receiver groups | Receiver sampling is usually small compared with assembly and solve cost. | Put pressure, velocity, DAS, and QC receiver groups in one simulation when they use the same model and sources. |
| Imaging | Forward and adjoint passes can reuse the same assembled system for a frequency. | Imaging workflows are most efficient when forward and adjoint work are scheduled together. |


## Imports

The tutorial uses the public `import frequensolve as fs` API. `pandas` is used only to display benchmark tables; the solver data and trace data are still read through FrequenSolve objects.


In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

import frequensolve as fs

u = fs.ureg


## A Small Model For Timing Experiments

The model is intentionally simple: two acoustic layers, a generated mesh, a source line near the top, and several receiver groups. The point is not to build a production survey; it is to isolate the cost of frequency sweeps, source-count scaling, and receiver sampling.

The helper below returns a complete simulation so each benchmark case is independent and saved under its own project directory. The material properties and mesh controls are kept fixed while the number of logical sources changes.


In [ ]:
def build_performance_simulation(
    *,
    name,
    path,
    n_sources=64,
    receiver_count=201,
):
    project = fs.Project(
        name="project",
        pretty_name="performance_tutorial",
        path=path,
        log_level="INFO",
        log_to_console=True,
    )

    sim = project.new_simulation(
        name=name,
        physics="acoustic",
        dimension=2,
        units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
    )

    model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.2])
    model.add_surface(name="top", depth=0.0 * u.km)
    model.add_layer(
        name="water",
        properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3},
    )
    model.add_surface(name="interface", depth=0.28 * u.km)
    model.add_layer(
        name="sediment",
        properties={"Vp": 2.4 * u.km / u.s, "Rho": 2.1 * u.g / u.cm**3},
    )
    model.add_surface(name="bottom", depth=0.7 * u.km)
    sim += model

    sim += model.hex_mesh_generator([10, 5])
    sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=4.0, f_high=28.0)
    sim.mesh.set_source_grading(d0=0.02, d1=0.10, factor=2.0)
    sim.mesh.set_receiver_grading(d0=0.02, d1=0.08, factor=2.0)

    sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
    sim += fs.BoundaryCondition(
        conditions=["pml"],
        boundaries=["x_min", "x_max", "z_max"],
        pml_wavelengths=0.75,
    )

    acq = fs.Acquisition()
    source_x = np.linspace(0.12, 1.08, n_sources)
    source_coords = fs.Q_([[x, 0.04] for x in source_x], "km")
    acq.add_source_group(kind="scalar", coords=source_coords)

    hydrophone = fs.ReceiverNode(name="hydrophone")
    hydrophone.add_component(name="p", field="pressure")

    dense_coords = fs.Q_([[x, 0.03] for x in np.linspace(0.05, 1.15, receiver_count)], "km")
    qc_coords = fs.Q_([[x, 0.12] for x in np.linspace(0.10, 1.10, 31)], "km")
    monitor_coords = fs.Q_([[0.6, z] for z in np.linspace(0.04, 0.50, 41)], "km")

    acq.add_receiver_group(name="surface_dense", device=hydrophone, coords=dense_coords)
    acq.add_receiver_group(name="surface_qc", device=hydrophone, coords=qc_coords)
    acq.add_receiver_group(name="vertical_monitor", device=hydrophone, coords=monitor_coords)
    sim += acq

    sim += fs.Discretization()
    sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)
    return project, sim


## Inspect The Model And Acquisition

This cell builds the baseline case with 64 logical sources. The receiver groups deliberately mix a dense production line, a smaller QC line, and a vertical monitor. Adding these receiver groups changes the amount of sampled output, but it does not create a new wave solve; the expensive work is still dominated by each frequency/source-batch solve.


In [ ]:
project_path = Path("./scratch/tutorials/performance")
project, sim = build_performance_simulation(
    name="performance_baseline",
    path=project_path,
    n_sources=64,
)

model = sim.model
model.plot("vp", figsize=(8, 3.2), aspect="equal")

{
    "sources": len(sim.acquisition.source_groups),
    "receiver_groups": [group.name for group in sim.acquisition.receiver_groups],
    "receiver_counts": {
        group.name: len(group.get_coordinates()) for group in sim.acquisition.receiver_groups
    },
}


## Start With A Frequency-Domain Sanity Check

A frequency-domain job is the fastest way to check that the model, mesh, acquisition, and boundary conditions are coherent. This job solves one frequency and requests ParaView output for visual QC. If this fails, there is no reason to launch a full time-domain sweep yet.

Job submission returns a future-like handle. Calling `.wait()` blocks the notebook until the submitted work completes and returns the completed job result. Section 2 covers local, cloud, and HPC sites in more depth; here we use a local site because the emphasis is performance diagnostics rather than deployment.


In [ ]:
site = fs.LocalSite(shutdown_on_completion=True, verbose=True)

fd_job = fs.FrequencyDomainJob(
    name="freq_qc",
    simulation=sim,
    f_list=[12.0],
)
fd_job += fs.ParaviewOutput(
    name="qc",
    path="paraview/qc",
    properties=["vp", "rho"],
    fields=["pressure"],
    sources=[1],
    show_pml=True,
)

fd_result = site.submit(fd_job).wait()
fd_job.print_frequency_summary()
fd_job.task_timings()


## Plot The Frequency-Domain QC Mesh Or Field

The VTU file can be opened directly in ParaView. The SDK also exposes a PyVista-backed convenience path for notebook QC. ParaView remains the preferred tool for detailed 3D inspection; the Python path is useful when you want a quick static screenshot in a notebook.


In [ ]:
vtu_files = fd_result.output_files(base="qc", suffix=".vtu", existing=True)
print("VTU outputs:")
for file in vtu_files:
    print(f"  {file}")

mesh_screenshot = project_path / "assets" / "performance_qc_vp.png"
mesh_screenshot.parent.mkdir(parents=True, exist_ok=True)

if vtu_files:
    fs.plot_vtu(
        vtu_files[0],
        field="vp",
        show_edges=True,
        scalar_bar=True,
        show=False,
        cmap=fs.BuGrOr,
        screenshot=mesh_screenshot,
        window_size=(1100, 500),
    )
    display(Image(filename=str(mesh_screenshot)))


## Run A Time-Domain Sweep

The time-domain job expands into many frequency tasks. Each task has its own run metadata, and the combined job writes a timing summary under the result directory. The helper methods used below read that metadata back into Python.

This is the same principle users should apply to production runs: launch a small frequency-domain QC first, then run the time-domain job once the setup is believable.


In [ ]:
time_job = fs.TimeDomainJob(
    name="time_sweep",
    simulation=sim,
    f_min=0.0,
    f_max=28.0,
    T_max=0.9,
)

time_result = site.submit(time_job).wait()
time_job.print_frequency_summary()


## Plot Per-Frequency Runtime

`task_timings()` gives one row per frequency task with total runtime. This is the first diagnostic to look at when a sweep feels slow: it shows whether cost is concentrated at a few frequencies or grows smoothly across the band.


In [ ]:
timing_rows = time_job.task_timings()
pd.DataFrame(timing_rows).head()

fig, ax = plt.subplots(figsize=(9, 3.8))
time_job.plot_task_timings(ax=ax, unit="seconds", max_xticks=10)
fig.tight_layout()


## Plot Assembly And Solve Phases

The solver timing manifest separates phases such as setup, mesh generation/adaptivity, matrix assembly, forward solve, adjoint solve, and imaging. For ordinary forward modeling, the dominant phases are usually assembly and forward solve. For imaging, forward and adjoint passes can use the same assembled system at a frequency, so assembly does not have to be repeated just because the adjoint solve is added.

The phase plot is a compact way to answer practical questions: are we spending time building the system, solving it, or writing/processing outputs?


In [ ]:
phase_order = ["setup", "mesh", "assembly", "solve_forward", "solve_adjoint", "imaging"]
phase_rows = time_job.phase_timings(phases=phase_order)
pd.DataFrame(phase_rows).head()

fig, ax = plt.subplots(figsize=(10, 4.2))
time_job.plot_phase_timings(
    ax=ax,
    phases=phase_order,
    include_zero=False,
    max_xticks=10,
    title="Time-domain sweep: solver phase timings by frequency",
)
fig.tight_layout()


## Source Batching In 2D

Source batching is solver-owned. The public acquisition describes logical sources and receivers; the solver/site decides how compatible sources are grouped during execution. Larger internal batches can amortize setup work, but they also increase memory pressure. In 2D, internal batches larger than 64 sources are split into sub-batches.

The calculation below mirrors that scheduling rule from the user's point of view: as the number of logical sources grows past 64, the solver needs more than one internal source batch.


In [ ]:
def expected_2d_source_batches(n_sources, internal_batch_cap=64):
    return {
        "sources": int(n_sources),
        "internal_2d_batch_cap": int(internal_batch_cap),
        "automatic_sub_batches": math.ceil(int(n_sources) / int(internal_batch_cap)),
    }

source_counts = [1, 8, 16, 32, 64, 96, 128]
pd.DataFrame([expected_2d_source_batches(n) for n in source_counts])


## Benchmark Source Count Scaling

This benchmark uses the same model and changes only the number of logical sources in the acquisition. The solver handles batching internally. The exact runtime curve depends on model size, solver settings, memory bandwidth, and site configuration, but this experiment shows whether adding compatible sources to one simulation scales smoothly on the current site.

The cell is intentionally strict: if a solver-side error occurs, the notebook stops and leaves logs/results for inspection.


In [ ]:
batch_site = fs.LocalSite(shutdown_on_completion=True, verbose=True)
batch_rows = []

for n_source_case in source_counts:
    _, batch_sim = build_performance_simulation(
        name=f"sources_{n_source_case:03d}",
        path=project_path,
        n_sources=n_source_case,
        receiver_count=101,
    )
    batch_job = fs.FrequencyDomainJob(
        name=f"sources_{n_source_case:03d}",
        simulation=batch_sim,
        f_list=[12.0],
    )
    batch_result = batch_site.submit(batch_job).wait()
    summary = batch_job.print_frequency_summary()
    timings = batch_job.task_timings()
    elapsed = timings[0]["duration_seconds"] if timings else np.nan
    batch_rows.append(
        {
            **expected_2d_source_batches(n_source_case),
            "succeeded": summary["succeeded"],
            "failed": summary["failed"],
            "duration_seconds": elapsed,
            "seconds_per_source": elapsed / n_source_case if elapsed else np.nan,
        }
    )

batch_table = pd.DataFrame(batch_rows)
batch_table


## Plot Source Scaling Results

The runtime curve shows total cost as more logical sources are included. The throughput curve, seconds per source, is often more useful: it should improve once the solver can amortize setup work across a useful internal batch, then flatten or step when the 2D internal cap creates additional sub-batches.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(
    batch_table["sources"],
    batch_table["duration_seconds"],
    marker="o",
)
axes[0].set_xlabel("Logical sources")
axes[0].set_ylabel("Single-frequency runtime (s)")
axes[0].set_title("Total runtime")
axes[0].grid(alpha=0.25)

axes[1].plot(
    batch_table["sources"],
    batch_table["seconds_per_source"],
    marker="o",
    color="tab:green",
)
for cap in sorted(batch_table["internal_2d_batch_cap"].unique()):
    axes[1].axvline(cap, color="0.4", linestyle="--", linewidth=1, alpha=0.5)
axes[1].set_xlabel("Logical sources")
axes[1].set_ylabel("Seconds per source")
axes[1].set_title("Throughput and internal 2D cap")
axes[1].grid(alpha=0.25)

fig.tight_layout()


## Receiver Groups Are Cheap Compared With Solves

The baseline simulation wrote three receiver groups from the same wavefield solve. This is the pattern to prefer when several measurements share the same source, model, and physics: add the receiver groups once, then select the group/component you need during analysis.

Adding receivers can increase output size and trace post-processing time, especially for dense DAS or many components, but it is usually much cheaper than launching separate simulations for each receiver layout.


In [ ]:
traces = time_result.traces(upscale=4)
traces.summary


In [ ]:
wavelet = fs.RickerWavelet(f=12.0)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)

for ax, group in zip(axes, ["surface_dense", "surface_qc", "vertical_monitor"]):
    gather = traces.td(group, "p", source=1, wavelet=wavelet, upscale=4, T_max=0.9)
    fs.plot_gather(gather, ax=ax, cmap="gray", title=group)

fig.tight_layout()


## Imaging Performance Note

In imaging workflows, the forward and adjoint equations at a frequency use the same model operator. FrequenSolve can therefore reuse the assembled system for the forward and adjoint passes instead of assembling twice. The phase timing plot exposes this directly: imaging jobs may show both `solve_forward` and `solve_adjoint`, while assembly remains a separate phase that can be shared.

This matters when estimating cost. A forward-only simulation and an imaging simulation are not related by simply doubling every phase; assembly and mesh work can be reused, while solve work grows with the number of right-hand sides and adjoint operations.


## Performance Checklist

Use this checklist before launching a large run.

| Check | Why it matters |
| --- | --- |
| Run one frequency first | Catches geometry, boundary condition, receiver, and output mistakes cheaply. |
| Inspect ParaView output | Confirms the mesh, PML, properties, and selected fields look sensible. |
| Print frequency summary | Shows succeeded, failed, and not-run frequency tasks immediately after the job. |
| Plot total task timings | Reveals expensive frequencies and failed/reused tasks. |
| Plot phase timings | Separates setup, mesh/adaptivity, assembly, solve, adjoint, and imaging cost. |
| Benchmark source batches | Start around 32-64 sources; confirm on the site where production will run. |
| Keep receiver groups together | Sampling multiple receiver layouts from one solve is usually much cheaper than rerunning the simulation. |
| Use frequency-domain QC before time-domain | Time-domain jobs are many frequency solves plus reconstruction, so they should not be the first test of a new setup. |
